# Chapter 10: From Grounded Evidence to a Defensible Report

The last stage of validation communicates the findings. The report must carry the evidence and its limits without distortion. Chapter 10 treats report generation as a further testing problem: the required content is fixed as an evidence packet and drafting is constrained by a closed-world writing contract. The delivered report is then checked against the store. This notebook applies the deterministic part of that procedure to the structural comparison of Chapter 5, whose measurements were confirmed in Chapter 9.

The notebook specifies the packet, checks complete and incomplete content plans, assembles the accepted content with source addresses and checks the delivered report for required facts and qualifications. It makes no language-model call. The recorded live drafting experiment on the same evidence is inspected in the live-run notebook `regional_live/ch10_report.ipynb`.

**Evidence.** The store is `book/evidence/parallel_aml_20260921/store.json`, whose regional records come from `book/evidence/aml_medoid_loop_leaf_output_20260921/`. The population is synthetic.

**Run order.** Run the cells from top to bottom. The coverage checks assert which plans are accepted and that the delivered report contains every required element.

## Setup and source checks

Use a Python kernel with the dependencies listed in the README. The setup cell locates the data root, which holds the evidence files and helper modules. It uses `MODEL_VALIDATION_ROOT` when that variable is set. Otherwise it searches for a repository checkout containing the current directory and then for the copy installed with the `aiv` package.

`load()` checks the digests that link the regional results, frozen finalists, protocol and admission records. The packet, qualification, plan-checking and assembly functions are defined in `companion.py` under the data root and use the Chapter 7 verifier.

In [1]:
from pathlib import Path
import os
import platform
import sys
import importlib.util

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
HELPERS = ROOT / "book/notebooks/regional_case"
for path in (ROOT, ROOT / "book", HELPERS):
    value = str(path)
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

# Load current helper files even if this kernel cached an older notebook revision.
def load_helper_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module

load_helper_module("companion", HELPERS / "companion.py")
load_helper_module("ch02_helpers", HELPERS / "ch02_helpers.py")
load_helper_module("weighted_leaf_geometry", ROOT / "book/weighted_leaf_geometry.py")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
from companion import RUN, read, load, frontier, confirmation, measure, assignments
pd.set_option("display.float_format", "{:.6f}".format)
protocol, results, frozen = load()
print("Evidence:", RUN.relative_to(ROOT).as_posix())
print("Mode: saved-evidence replay; no fitting or live LLM calls")


Python: 3.12.3
Data root found from: repository checkout
Evidence: book/evidence/aml_medoid_loop_leaf_output_20260921
Mode: saved-evidence replay; no fitting or live LLM calls


## Specify the packet

A validation report can be accurate sentence by sentence and still mislead, by omitting an unfavorable result or a condition on which a finding depends. Chapter 10 addresses this by fixing, before drafting, what a report section must contain (Section 10.1). An evidence packet lists the required fact identifiers from the store and the required qualifications, the statements of scope and limitation that must accompany those facts (Section 10.3).

`report_packet()` builds the packet for a section reporting the Chapter 5 comparison. It requires the confirmation sample size and, for each of the three seeds, the Brier loss, share and contrast of both the continuation and the reflection regions together with their paired difference and interval. `QUALIFIERS` holds the six required qualifications, each keyed by a short name. The cell prints the counts and displays the packet and the qualification texts.

This notebook assembles a report deterministically from authored content plans. It does not replay or generate language-model drafts; the live drafting experiment on the same evidence is inspected in the live-run notebook named at the end.

In [2]:
from companion import report_packet, QUALIFIERS, validate_plan, render_report
store,packet=report_packet()
print("Required facts:",len(packet["required_facts"]))
print("Required qualifications:",len(packet["required_qualifiers"]))
display(packet)
display(QUALIFIERS)

Required facts: 25
Required qualifications: 6


{'required_facts': ['region:protocol/confirmation_n',
  'region:confirmation/continuation_73101/brier',
  'region:confirmation/continuation_73101/share',
  'region:confirmation/continuation_73101/contrast',
  'region:confirmation/reflection_73101/brier',
  'region:confirmation/reflection_73101/share',
  'region:confirmation/reflection_73101/contrast',
  'region:comparison/reflection-continuation_73101/contrast_difference',
  'region:comparison/reflection-continuation_73101/simultaneous_ci',
  'region:confirmation/continuation_73102/brier',
  'region:confirmation/continuation_73102/share',
  'region:confirmation/continuation_73102/contrast',
  'region:confirmation/reflection_73102/brier',
  'region:confirmation/reflection_73102/share',
  'region:confirmation/reflection_73102/contrast',
  'region:comparison/reflection-continuation_73102/contrast_difference',
  'region:comparison/reflection-continuation_73102/simultaneous_ci',
  'region:confirmation/continuation_73103/brier',
  'region:co

{'synthetic': 'These measurements concern simulated account-months and do not establish operational AML adequacy.',
 'fixed': 'The logistic predictor and auxiliary error model remained frozen; search changed region membership.',
 'conditional': 'Intervals are approximate simultaneous 95% normal intervals for 19 claims, conditional on the fitted models and frozen regions.',
 'replication': 'Three optimizer seeds share one discovery sample, one LLM proposal and one confirmation sample.',
 'comparison': 'All three reflection-minus-continuation intervals include zero; a positive advantage is unresolved.',
 'coverage': 'The selected regions differ in coverage; contrast is not predictor repair.'}

**Read the output.** The packet requires 25 facts: one sample size and eight measurements for each of three seeds. The continuation results are required alongside the reflection results, so a section cannot report the structural branch without its control. Every seed is required, including seed 73102, where the structural branch has the lower contrast. The six qualifications state that the population is synthetic, that the models were frozen, that the intervals are conditional simultaneous intervals for 19 claims and that the seeds share their samples and proposal. The last two state that all three structural comparisons include zero and that contrast is not predictor repair.

## Check proposed coverage

A content plan is the list of fact identifiers and qualification names that a drafter proposes to include. The plan can be checked before any text is drafted. `validate_plan(store, packet, plan)` rejects a plan with duplicate entries, rejects any plan whose facts or qualifications differ from the packet's (whether by omission or by an unrequested addition) and verifies every planned fact against the store with the Chapter 7 verifier.

The cell checks three authored plans: a complete plan, a plan missing the final fact (the seed-73103 paired interval) and a plan missing the `comparison` qualification. No plan is attributed to a language model.

In [3]:
from copy import deepcopy
complete={"facts":packet["required_facts"].copy(),"qualifiers":packet["required_qualifiers"].copy()}
missing_fact=deepcopy(complete);missing_fact["facts"].pop()
missing_scope=deepcopy(complete);missing_scope["qualifiers"].remove("comparison")
rows=[]
for name,plan in [("complete",complete),("missing interval",missing_fact),("missing comparison qualifier",missing_scope)]:
    valid,reason=validate_plan(store,packet,plan)
    rows.append(dict(plan=name,accepted=valid,reason=reason))
assert [r["accepted"] for r in rows]==[True,False,False]
with pd.option_context("display.max_colwidth",None):
    display(pd.DataFrame(rows))

,plan,accepted,reason
0,complete,True,Complete source-checked packet
1,missing interval,False,Missing or unexpected evidence/qualification identifiers
2,missing comparison qualifier,False,Missing or unexpected evidence/qualification identifiers


**Read the output.** The complete plan is accepted as a complete source-checked packet, and both incomplete plans are rejected for missing identifiers. The second rejection matters most for interpretation. Without the `comparison` qualification, a section could report the three positive or small differences and the admitted union rule without stating that no advantage was resolved. The check enforces a declared contract. It does not measure whether a drafter, human or model, would propose a complete plan unprompted.

## Assemble accepted content

Once a plan is accepted, the section can be assembled from the store without retyping any number. The assembly function in the cell validates the plan again. It then writes one line per fact, giving the identifier, the stored value and its source file and JSON pointer, followed by the full text of each required qualification. The values are written unrounded, as stored; a formatted report would round them for display while keeping the addresses.

In [4]:
report=render_report(store,packet,complete)
print(report)

Regional discovery report: deterministic assembly from an authored content plan.

region:protocol/confirmation_n: 200000 [source: book/evidence/aml_medoid_loop_leaf_output_20260921/protocol.json#/confirmation_n]
region:confirmation/continuation_73101/brier: 0.07258430947407625 [source: book/evidence/aml_medoid_loop_leaf_output_20260921/results.json#/confirmation/continuation_73101/brier]
region:confirmation/continuation_73101/share: 0.113925 [source: book/evidence/aml_medoid_loop_leaf_output_20260921/results.json#/confirmation/continuation_73101/share]
region:confirmation/continuation_73101/contrast: 0.057939624355884026 [source: book/evidence/aml_medoid_loop_leaf_output_20260921/results.json#/confirmation/continuation_73101/contrast]
region:confirmation/reflection_73101/brier: 0.07285078874096361 [source: book/evidence/aml_medoid_loop_leaf_output_20260921/results.json#/confirmation/reflection_73101/brier]
region:confirmation/reflection_73101/share: 0.11522 [source: book/evidence/aml_m

**Read the output.** Each line binds a value to its source, for example the continuation contrast at seed 73101, 0.057939624355884026, to `results.json#/confirmation/continuation_73101/contrast`. A reviewer can therefore check any line against the evidence with the Chapter 6 resolution. The assembled text is a structured record of what the section must say. Turning it into readable paragraphs is the drafting step that Chapter 10 assigns to a language model under a closed-world writing contract (Section 10.4), after which the draft must be checked again.

## Check the deliverable

A report is checked as delivered, not only as planned (Section 10.10). The cell counts how many required fact identifiers and qualification texts appear in the assembled report and asserts that all are present. It then prints the comparison qualification, the statement that carries the study's central negative result.

In [5]:
facts_found=sum(id_ in report for id_ in packet["required_facts"])
qualifiers_found=sum(QUALIFIERS[q] in report for q in packet["required_qualifiers"])
print(f"Fact identifiers present: {facts_found} of {len(packet['required_facts'])}")
print(f"Qualifications present: {qualifiers_found} of {len(packet['required_qualifiers'])}")
assert facts_found==len(packet["required_facts"]) and qualifiers_found==len(packet["required_qualifiers"])
print("Comparison qualification:",QUALIFIERS["comparison"])

Fact identifiers present: 25 of 25
Qualifications present: 6 of 6
Comparison qualification: All three reflection-minus-continuation intervals include zero; a positive advantage is unresolved.


**Read the output.** All 25 identifiers and all six qualifications are present, and the comparison qualification states that all three reflection-minus-continuation intervals include zero. Presence of identifiers is a coverage check. It shows that required evidence is cited, but it cannot show that the sentences built around those citations interpret them correctly. In the live drafting experiment inspected in `regional_live/ch10_report.ipynb`, six of nine language-model drafts passed identifier coverage, while a separate semantic review found that seven of the nine needed substantive revision.

## What the replay establishes

A report section on the structural comparison has been specified as a packet of 25 facts and six qualifications, checked as a content plan, assembled with source addresses and checked as delivered. Plans that omit an interval or the qualification carrying the negative result are rejected before any text is drafted.

These checks enforce coverage and source binding. They do not assess whether generated text states the evidence faithfully, which requires the semantic review that the live experiment shows to be necessary.


## What to try

1. **An omitted control.** Remove `region:confirmation/continuation_73102/contrast` from the complete plan and confirm that it is rejected. Explain how the section's interpretation could change without that control.
2. **An unrequested fact.** Add `region:discovery/reflection_73101/region/contrast` to the complete plan. Predict the outcome before running it, and explain why an extra discovery value in a confirmation section is a reporting risk.
3. **A faithful sentence.** Write one sentence summarizing seed 73102 from the assembled lines. Check each number in it with `verify`, then explain what the verifier cannot check about the sentence.
4. **A live evaluation.** Specify an Inner Array of two reporting instructions and an Outer Array of two reserved packets for a live drafting study. State which outcomes would be measured automatically and which would require review.
